# RV32I-MLA CPU Bring-Up & Micro-Diagnostics
## Frozen 3-stage CPU validation notebook

This notebook documents the bring-up and validation of a 3-stage RV32I CPU on PYNQ.

In [ ]:
from pynq import Overlay, MMIO
import time

BIT_NAME = "rv32i_mla_bd.bit"


In [ ]:
def whex(x):
    return f"0x{(x & 0xFFFFFFFF):08X}"


In [ ]:
IMEM_BASE = 0x0000
DMEM_BASE = 0x0100


In [ ]:
def rv_addi(rd, rs1, imm): return ((imm & 0xFFF)<<20)|(rs1<<15)|(0<<12)|(rd<<7)|0b0010011
def rv_add(rd, rs1, rs2): return (0<<25)|(rs2<<20)|(rs1<<15)|(0<<12)|(rd<<7)|0b0110011
def rv_sw(rs2, rs1, imm): return ((imm>>5)<<25)|(rs2<<20)|(rs1<<15)|(2<<12)|((imm&0x1F)<<7)|0b0100011
def rv_jal(rd, imm): return (((imm>>20)&1)<<31)|(((imm>>12)&0xFF)<<12)|(((imm>>11)&1)<<20)|(((imm>>1)&0x3FF)<<21)|(rd<<7)|0b1101111


In [ ]:
prog_no_branch = [
    rv_addi(1,0,5),
    rv_addi(2,0,7),
    rv_add(3,1,2),
    rv_sw(3,0,0x100),
    rv_addi(5,0,2),
    rv_sw(5,0,0x104),
    rv_jal(0,0),
]


In [ ]:
ol = Overlay(BIT_NAME)
mmio = MMIO(list(ol.mem_dict.values())[0]["phys_addr"],
            list(ol.mem_dict.values())[0]["addr_range"])


In [ ]:
for i, w in enumerate(prog_no_branch):
    mmio.write(IMEM_BASE + 4*i, w)

time.sleep(0.05)

d0 = mmio.read(DMEM_BASE + 0)
d1 = mmio.read(DMEM_BASE + 4)

print("DMEM[0x100] =", whex(d0))
print("DMEM[0x104] =", whex(d1))
